### Required Files/Directories

- **graph_credentials.json -** This file should be present under _`datahub-fork\datahub-integrations-service\experiments\docs_generation/`_ directory. It contains tokens for all instances
- **resource_files -** This directory needs to be present in _`datahub-fork\datahub-integrations-service\experiments\term_suggestion_v2\`_, it contains following files
  - **test_urns.json** - This file contains table urns for all instances
  - **.env** - Specify values of environment variables BEDROCK_AWS_ACCESS_KEY_ID and BEDROCK_AWS_SECRET_ACCESS_KEY
  - **care_columns_ground_truth.csv** - Ground truth CSV for Care URNs
  - **column_labels_without_care_data_final** - Ground truth CSV for Non-Care URNs
  - **care_glossary_terms_with_term_descriptions_and_examples.pkl -** Contains Care terms descriptions generated externally. This file can be generated using _"populate_glossary_terms_examples_and_descriptions.ipynb"_
  - **CARE_COLUMNS_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL -** This file contains description for CARE instance columns generated using internal API. This file can be generated using _generate_descriptions_for_columns.ipynb_
  - **CARE_TABLES_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL -** This file contains description for CARE instance tables generated using internal API. This file can be generated using _generate_descriptions_for_columns.ipynb_
- **prompts -** This directory is intended to contain all experimental prompts in a text file. This directory can be placed in _`datahub-fork\datahub-integrations-service\experiments\term_suggestion_v2\`_. This directory is checked in to Git.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import pathlib
import dotenv
import pickle
import sys
import json5
import pandas as pd
import numpy as np
from typing import Dict, List
from datetime import datetime

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger

# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)

from docs_generation.graph_helper import create_datahub_graph
from helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict,
    get_failed_response_table_urns,
    read_labeled_column_data,
    get_prediction_df,
    filter_predictions_df,
    get_classification_report_df,
)


dotenv.load_dotenv(current_dir / "resource_files/.env")

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class ExperimentConfig:
    prompt_path: str
    experiment_name: str
    iterations: int
    confidence_thresholds: List[float]


experiment_dict = [
    ExperimentConfig(
        prompt_path="prompts/FN_examples_reformatted.txt",
        experiment_name="FN_reformatted_examples_term_desc",
        iterations=3,
        confidence_thresholds=[7, 8, 8.5, 9, 9.5, 10],
    ),
    ExperimentConfig(
        prompt_path="prompts/FN_examples_user_id_inst.txt",
        experiment_name="FN_examples_term_desc_user_id_inst",
        iterations=3,
        confidence_thresholds=[7, 8, 8.5, 9, 9.5, 10],
    ),
]

In [ ]:
# Experiment settings:
EXAMPLES_IN_GLOSSARY_TERMS = False
USE_TERM_DESCRIPTIONS = True
USE_TERM_PARENT_INFO = True

USE_COLUMN_INFO_WITH_DESCRIPTIONS = False
OLD_LABELS = False

BASE_DIRECTORY = (
    current_dir / f"output/output_{datetime.now().strftime('%m-%d-%H-%M')}/"
)
print(BASE_DIRECTORY)
RESOURCE_DIR = current_dir / "resource_files/"
COLUMN_LABELS_CSV_PATH = RESOURCE_DIR / "care_columns_ground_truth.csv"
GLOSSARY_TERMS_INFO_FILE_PATH = (
    RESOURCE_DIR / "care_glossary_terms_with_term_descriptions_and_examples.pkl"
)

URNS_DICT_PATH = RESOURCE_DIR / "test_urns.json"
INSTANCES_TO_EXAMINE = ["care"]
GLOSSARY_INSTANCE = "care"
GLOSSARY_NODES = [
    "urn:li:glossaryNode:d966ad51-49a7-411c-8d06-2ee2dd210647",
    "urn:li:glossaryNode:09effdec-8810-415c-ace7-4e53090c502c",
]
GLOSSARY_TERMS = []


if OLD_LABELS:
    LABEL_COLUMN = "glossary_terms_updated"
else:
    LABEL_COLUMN = "glossary_terms_updated_new"

URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
    URNS_DICT_PATH.read_text()
)


FILTERS = [
    "no_filter",
    # "description",
    # "sample_values",
    # "description_and_sample_values",
    # "description_or_sample_values",
    # "name_and_datatype",
]
URNS_DICT = {
    key: value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE
}

In [ ]:
if USE_COLUMN_INFO_WITH_DESCRIPTIONS:
    TABLE_INFO_FILE_PATH = (
        RESOURCE_DIR / "CARE_TABLES_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL"
    )
    COLUMN_INFO_FILE_PATH = (
        RESOURCE_DIR / "CARE_COLUMNS_INFO_WITH_AUTO_GENERATED_DESCRIPTIONS.PKL"
    )
    with open(TABLE_INFO_FILE_PATH, "rb") as f:
        tables_info_dict = pickle.load(f)
        f.close()
    with open(COLUMN_INFO_FILE_PATH, "rb") as f:
        columns_info_dict = pickle.load(f)
        f.close()
else:
    tables_info_dict, columns_info_dict = get_table_and_column_infos_dict(
        urns_dict=URNS_DICT
    )

# Fetch Glossary terms


In [ ]:
if not EXAMPLES_IN_GLOSSARY_TERMS and not USE_TERM_DESCRIPTIONS:
    print("fetching terms......")
    graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
    glossary_config = GlossaryUniverseConfig(
        glossary_nodes=GLOSSARY_NODES, glossary_terms=GLOSSARY_TERMS
    )
    glossary_info = fetch_glossary_info(
        graph_client=graph_client, universe=glossary_config
    )
else:
    print("loading terms......")
    with open(GLOSSARY_TERMS_INFO_FILE_PATH, "rb") as f:
        glossary_info = pickle.load(f)

if not EXAMPLES_IN_GLOSSARY_TERMS:
    print("removing examples......")
    for key, value in glossary_info.glossary.items():
        glossary_info.glossary[key].pop("example", None)

if not USE_TERM_PARENT_INFO:
    print("removing parent node info ......")
    for key, value in glossary_info.glossary.items():
        glossary_info.glossary[key].pop("parent_node", None)

# print(len(glossary_info.glossary.keys()))
# terms_to_remove = ['Day', 'Month', 'Year']
# glossary_info.glossary = {key: value for key, value in glossary_info.glossary.items() if value["term_name"] not in terms_to_remove}
# print(len(glossary_info.glossary.keys()))

In [ ]:
terms = []
for key, value in glossary_info.glossary.items():
    terms.append(value["term_name"])
print(len(terms))
print(terms)

In [ ]:
def func_categorize(row):
    if len(row["pred_max_score_term"]) == 0:
        if row[LABEL_COLUMN] is None:
            return "match-no_assignment"
        else:
            return "mismatch-actual_only"
    elif row[LABEL_COLUMN] is None:
        return "mismatch-predicted_only"
    else:
        actual_terms = [term.strip() for term in row[LABEL_COLUMN].split("/")]
        if len(np.intersect1d(actual_terms, row["pred_max_score_term"])) == 0:
            return "mismatch"
        else:
            return "match-term_assigned"


def check_match(row):
    #     print(row)
    if (
        (row["glossary_term"] is None and len(row["pred_max_score_term"]) == 0)
        or row["glossary_term"] in row["pred_max_score_term"]
        or row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"]
    ):
        return 1
    else:
        return 0


def get_merged_prediction_df(labeled_df, df_pred):
    merged_df = pd.merge(
        labeled_df,
        df_pred,
        on="unique_keys",
        how="left",
    )
    merged_df.loc[:, "pred_max_score_term"] = merged_df["pred_max_score_term"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    merged_df.loc[:, "predicted_labels"] = merged_df["predicted_labels"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    not_omitted_columns_df = merged_df[
        merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]
    omitted_columns_df = merged_df[
        ~merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]

    #     merged_df.loc[:, "match"] = merged_df.apply(lambda x: check_match(x), axis=1)
    merged_df.loc[:, "label_class"] = merged_df.apply(
        lambda x: func_categorize(x), axis=1
    )
    return merged_df, not_omitted_columns_df, omitted_columns_df


def get_parsed_responses_for_single_experiment_run(
    urns_dict, glossary_info, prompt_path
):
    raw_llm_responses = []
    parsed_llm_responses = []

    for instance, urns in urns_dict.items():
        graph_client = create_datahub_graph(instance)
        for urn in urns:
            print(urn)
            try:
                table_terms, column_terms, raw_llm_response = get_term_recommendations(
                    table_urn=urn,
                    graph_client=graph_client,
                    glossary_info=glossary_info,
                    prompt_path=prompt_path,
                )
                #             column_terms = label_fake_terms(column_terms)
                raw_llm_responses.append([urn, instance, raw_llm_response])
                parsed_llm_responses.append((urn, instance, table_terms, column_terms))
            except Exception as e:
                logger.exception(f"Exception Occurred {e}")
                raw_llm_responses.append([urn, instance, raw_llm_response])
                parsed_llm_responses.append((urn, instance, None, None))
                continue
    return parsed_llm_responses, raw_llm_responses


def get_aggregated_metrics(classification_report_df, confidence_threshold):
    total_TP = classification_report_df.TP.sum()
    total_FP1 = classification_report_df.FP1.sum()
    total_FP2 = classification_report_df.FP2.sum()
    total_FN = classification_report_df.FN.sum()
    total_TN = classification_report_df.TN.sum()

    precision_term_for_all_tables = (
        total_TP / (total_TP + total_FP1 + total_FP2)
        if (total_TP + total_FP1 + total_FP2) != 0
        else np.nan
    )

    recall_term_for_all_tables = (
        total_TP / (total_TP + total_FN + total_FP2)
        if (total_TP + total_FN + total_FP2) != 0
        else np.nan
    )

    precision_none_for_all_tables = (
        total_TN / (total_TN + total_FN) if (total_TN + total_FN) != 0 else np.nan
    )

    recall_none_for_all_tables = (
        total_TN / (total_TN + total_FP1) if (total_TN + total_FP1) != 0 else np.nan
    )

    aggregated_metrics = {
        "precision_term_for_all_tables": precision_term_for_all_tables,
        "recall_term_for_all_tables": recall_term_for_all_tables,
        "precision_none_for_all_tables": precision_none_for_all_tables,
        "recall_none_for_all_tables": recall_none_for_all_tables,
        "threshold": confidence_threshold,
    }
    return aggregated_metrics


def get_final_statistics_for_confidence_threshold(
    merged_prediction_df,
    parsed_llm_responses,
    filters,
    columns_info_dict,
    labeled_df,
    confidence_threshold,
):
    final_stats = {}
    classification_reports = {}
    for filter_ in filters:
        filtered_prediction_df = filter_predictions_df(merged_prediction_df, filter_)
        classification_report_df = get_classification_report_df(
            parsed_llm_responses=parsed_llm_responses,
            pred_df=filtered_prediction_df,
            column_infos_dict=columns_info_dict,
            labeled_df=labeled_df,
            filter_=filter_,
        )
        classification_reports[filter_] = classification_report_df
        temp_stats = dict(
            classification_report_df[
                [
                    "fake_columns_count",
                    "fake_terms_count",
                    "actual_column_count",
                    "skipped_columns_count",
                    "TP",
                    "TN",
                    "FP1",
                    "FP2",
                    "FN",
                ]
            ].sum(axis=0)
        )
        aggregated_metrics = get_aggregated_metrics(
            classification_report_df=classification_report_df,
            confidence_threshold=confidence_threshold,
        )
        temp_stats.update(aggregated_metrics)
        final_stats[filter_] = temp_stats
    final_stats_df = pd.DataFrame(final_stats)
    return final_stats_df, classification_reports


def save_misclassification_analysis(
    merged_prediction_df, dest_dir, confidence_threshold
):
    miscalssification_analysis_cols = [
        "table_urn",
        "col_name",
        "col_description",
        "sample_values",
        LABEL_COLUMN,
        "predicted_labels",
    ]

    merged_prediction_df[merged_prediction_df.label_class == "mismatch-actual_only"][
        miscalssification_analysis_cols
    ].to_csv(dest_dir / f"FN_threshold_{confidence_threshold}.csv")

    merged_prediction_df[merged_prediction_df.label_class == "mismatch-predicted_only"][
        miscalssification_analysis_cols
    ].to_csv(dest_dir / f"FP1_threshold_{confidence_threshold}.csv")

    merged_prediction_df[merged_prediction_df.label_class == "mismatch"][
        miscalssification_analysis_cols
    ].to_csv(dest_dir / f"FP2_threshold_{confidence_threshold}.csv")
    return None


def populate_analysis_for_confidence_threshold_list(
    output_dir,
    parsed_llm_responses,
    filters,
    urns_dict,
    columns_info_dict,
    confidence_thresholds,
):
    for confidence_threshold in confidence_thresholds:
        print("CONFIDENCE_THRESHOLD", confidence_threshold)

        # Make Directory
        SUB_DIR = output_dir / f"threshold_{confidence_threshold}"
        SUB_DIR.mkdir(parents=True, exist_ok=True)

        # Read labelled data
        labeled_df = read_labeled_column_data(COLUMN_LABELS_CSV_PATH, urns_dict)
        tables_with_parsing_error, skipped_tables = get_failed_response_table_urns(
            parsed_llm_responses
        )
        print("tables_with_parsing_error:", tables_with_parsing_error)
        print("skipped_tables:", skipped_tables)

        # Prepare prediction df:
        df_pred = get_prediction_df(parsed_llm_responses, confidence_threshold)
        print("labeled_df", len(labeled_df))
        print("prediction_df", len(df_pred))

        # Merge Prediction df and labelled df
        merged_df, _, _ = get_merged_prediction_df(
            labeled_df[~labeled_df.table_urn.isin(tables_with_parsing_error)],
            df_pred,
        )
        print("merged_df", len(merged_df))
        merged_df.to_csv(
            SUB_DIR / f"predicted_labels_threshold_{confidence_threshold}.csv"
        )

        # Misclassification Analysis:
        save_misclassification_analysis(merged_df, SUB_DIR, confidence_threshold)

        # Final Statistics
        final_stats_df, classification_reports = (
            get_final_statistics_for_confidence_threshold(
                merged_prediction_df=merged_df,
                parsed_llm_responses=parsed_llm_responses,
                filters=filters,
                columns_info_dict=columns_info_dict,
                labeled_df=labeled_df,
                confidence_threshold=confidence_threshold,
            )
        )

        for key, classification_report_df in classification_reports.items():
            classification_report_df.to_csv(SUB_DIR / f"{key}_cf_df.csv")
        final_stats_df.to_csv(
            SUB_DIR / f"final_stats_threshold_{confidence_threshold}.csv"
        )
    return None

In [ ]:
from tqdm import tqdm


def write_experiment_details():
    with open(BASE_DIRECTORY / "experiment_details.txt", "w") as f:
        f.write(
            f"EXAMPLES_IN_GLOSSARY_TERMS: {EXAMPLES_IN_GLOSSARY_TERMS}\n"
            f"USE_COLUMN_INFO_WITH_DESCRIPTIONS: {USE_COLUMN_INFO_WITH_DESCRIPTIONS}\n"
            f"USE_TERM_DESCRIPTIONS: {USE_TERM_DESCRIPTIONS}\n"
            f"OLD_LABELS: {OLD_LABELS}"
        )
        f.close()
    # with open(BASE_DIRECTORY / "glossary_info.pkl", "wb") as f:
    #     pickle.dump(glossary_info, f)
    # with open(BASE_DIRECTORY / "tables_info.pkl", "wb") as f:
    #     pickle.dump(tables_info_dict, f)
    # with open(BASE_DIRECTORY / "columns_info.pkl", "wb") as f:
    #     pickle.dump(columns_info_dict, f)
    return None

In [ ]:
for experiment in tqdm(experiment_dict, total=len(experiment_dict)):
    (BASE_DIRECTORY / f"{experiment.experiment_name}").mkdir(
        parents=True, exist_ok=True
    )
    for run in range(experiment.iterations):
        print(f"run_{run}")
        # Create Output Directory:
        OUTPUT_DIR = (
            BASE_DIRECTORY
            / f"{experiment.experiment_name}"
            / f"run{run+1}_{datetime.now().strftime('%H-%M-%S')}"
        )
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # write experiment details:
        write_experiment_details()

        # read and save prompt:
        prompt_path = current_dir / experiment.prompt_path
        prompt = prompt_path.read_text()
        pathlib.Path(OUTPUT_DIR / "prompt.txt").write_text(prompt)

        # Generate Response:
        parsed_llm_responses, raw_llm_responses = (
            get_parsed_responses_for_single_experiment_run(
                urns_dict=URNS_DICT,
                glossary_info=glossary_info,
                prompt_path=prompt_path,
            )
        )

        # Write response to directory
        write_llm_output_to_csv(
            llm_response=parsed_llm_responses,
            csv_path=OUTPUT_DIR / "parsed_response.csv",
        )
        with open(OUTPUT_DIR / "parsed_response.pkl", "wb") as fp:
            pickle.dump(parsed_llm_responses, fp)

        # Populate Analysis
        populate_analysis_for_confidence_threshold_list(
            output_dir=OUTPUT_DIR,
            confidence_thresholds=experiment.confidence_thresholds,
            filters=FILTERS,
            urns_dict=URNS_DICT,
            parsed_llm_responses=parsed_llm_responses,
            columns_info_dict=columns_info_dict,
        )

In [ ]:
# SKIP: Only for testing purposes.
# LABEL_COLUMN = "glossary_term"
# populate_analysis_for_confidence_threshold_list(
#     output_dir=OUTPUT_DIR,
#     confidence_thresholds=experiment.confidence_thresholds,
#     filters=FILTERS,
#     urns_dict=URNS_DICT,
#     parsed_llm_responses=parsed_llm_responses,
#     columns_info_dict=columns_info_dict,
# )

In [ ]:
print("Done")

     **actual**            **predicted**           **cateory_name**
       None                    None                  match-no_assignment              TN
       None                    term                  mismatch-predicted_only          FP1
       term                    different term        mismatch                         FP2
       term                    term                  match-term_assigned              TP
       term                    None                  mismatch-actual_only             FN

    Recall of positive class:  TP/(TP + FN + FP2)
    Precision : TP/(TP + FP1 + FP2)

    Recal of negative class: TN/(TN + FP1)
    Precision of negative class: TN/(TN + FN)
